# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR^2) Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process a FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source

The dataset is described by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

> **Note:** All entities (record sets, fields, columns, etc.) are referenced by their `@id` as per the Croissant specification and this template.

In [ ]:
# Ensure `mlcroissant` library is installed (uncomment if not already installed)
!pip install mlcroissant

## 1. Data Loading

This section loads metadata and records from the dataset using the [`mlcroissant`](https://mlcroissant.readthedocs.io/) library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata and instantiate a Croissant Dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset metadata (access as object attributes)
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview

Enumerate available record sets, their `@id` values, and fields within each record set by `@id`. Use these identifiers for further data extraction and manipulation.

This step helps to understand how the table(s) and field(s) are organized following the Croissant schema.

In [ ]:
# List all record sets and their fields (`@id` values)

print("Available record sets and fields by @id:")
record_sets = list(dataset.record_sets)
for record_set in record_sets:
    print(f"- Record set: {record_set['@id']}")
    if 'fields' in record_set:
        print("  Fields:")
        for field in record_set['fields']:
            if hasattr(field, 'to_json'):
                field_id = field.to_json().get('@id', None)
            elif isinstance(field, dict):
                field_id = field.get('@id', None)
            else:
                field_id = str(field)
            print(f"    - {field_id}")
    else:
        print("  (No fields detected)")
    print()

## 3. Data Extraction

Load data from each record set into a pandas DataFrame for downstream processing. We use the `@id` values identified above.

> **Tip:** Replace or adapt the `record_set_ids` list for your workflow to select only needed tables.

In [ ]:
# Extract all record set @id values
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set: {record_set_id} (shape: {dataframes[record_set_id].shape})")
    else:
        print(f"No records found for record set: {record_set_id}")

# Show columns of the main clinical data record set if available
if dataframes:
    main_record_set_id = record_set_ids[0]
    print(f"\nColumns in DataFrame for: {main_record_set_id}")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps for EDA. Modify the field and record set `@id` as appropriate.

Sample operations below include:
- Filtering records based on a numeric field.
- Z-score normalization of a numeric variable.
- Grouping by a categorical group field.

🚩 **Make sure to adjust the field `@id`s based on the printed columns above!**

In [ ]:
# --- Edit these as needed after reviewing the printed fields/columns ---

# Sample: use '@id' of main record set and a numeric field
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]

# Example field IDs (update to actual field @id strings from data overview):
numeric_field_id = None
group_field_id = None

# Try to guess a numeric field by searching for typical names
for col in df.columns:
    if any(x in col.lower() for x in ['age', 'interval', 'years', 'duration']):
        numeric_field_id = col
        print(f"Using numeric field: {numeric_field_id}")
        break
if not numeric_field_id:
    numeric_field_id = df.select_dtypes(include='number').columns[0]
    print(f"Defaulting to first numeric field: {numeric_field_id}")

# Try to select a grouping field (categorical)
for col in df.columns:
    if any(x in col.lower() for x in ['sex', 'gender', 'site', 'location', 'histology']):
        group_field_id = col
        print(f"Using group field: {group_field_id}")
        break


# 1. Filtering by numeric field
threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else None
if threshold is not None:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (N={len(filtered_df)}):")
    display(filtered_df.head())

    # 2. Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - df[numeric_field_id].mean()) / df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' (z-score):")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # 3. Grouping (if possible)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped data by '{group_field_id}':")
        display(grouped_df.head())
else:
    print(f"No suitable numeric field found for filtering in record set '{main_record_set_id}'.")

## 5. Visualization

Visualize basic data distributions or relationships using matplotlib or seaborn.

Edit the field IDs and visualizations as needed for your analysis.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))

if numeric_field_id in df.columns:
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

if group_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion

In this notebook, you loaded and explored the FAIR^2 second primary colorectal cancer dataset via its Croissant schema using `mlcroissant`.

- We inspected schema-driven `@id` references for record sets and fields.
- We extracted all available records and loaded the main data to pandas DataFrames.
- We performed basic filtering, normalization, grouping, and visualizations, referencing all fields and sets by their `@id` for reproducible data exploration.

**Next steps:** Explore more advanced modeling, link additional Croissant-defined semantics, or customize EDA steps based on research questions.